# Epidemic curves from a JUNE2 events file (events-only)

The platform's **tracer bullet**: load an events file -> aggregate over time -> plot an epidemic curve, using **only** `simulation_events.h5` (no `world_state.h5`, no coordinates). Proves `core/` is reusable without any rendering-of-geography (ADR-0003) and without render deps below the core boundary (ADR-0002 -- matplotlib is imported *here*, in the consumer, never in `core`).

Requires `requirements.txt` + `requirements-plotting.txt` (and optionally `requirements-aggregate.txt` for the numba speed-up).

## Parameters
Point `events_path` at any `simulation_events.h5`. The default is an example full-England run; swap it for the small committed example fixture when one is added -- the structure is identical.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `core` resolves (no packaging yet).
repo_root = Path.cwd()
while not (repo_root / "core").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# --- parameters ---
events_path = repo_root.parent / "JUNE2" / "runs" / "run_full_england_modern" / "simulation_events.h5"
event_types = ["infections", "deaths"]  # which curves to plot
days_per_bin = 1.0

print("repo_root :", repo_root)
print("events_path:", events_path, "(exists:", events_path.exists(), ")")

## Auto-discover available event types
`inspect_file` lists every dataset in the file; event types live under `events/`. No event type is hardcoded.

In [ ]:
from core.load_data.june_events import inspect_file, load_enriched_events
from core.aggregate import aggregate_events, epidemic_curve

summary = inspect_file(str(events_path))
available_event_types = sorted(
    dataset.path.split("/", 1)[1]
    for dataset in summary.datasets
    if dataset.path.startswith("events/") and dataset.n_rows > 0
)
print("available event types:", available_event_types)

## Load, aggregate, build curves
For each chosen event type: load the **enriched** table (events + lookup joins, so each row already carries `geo_unit_id`), aggregate into dense per-geo per-day counts keyed on `geo_unit_id`, then sum over geo units to get the epidemic curve.

In [ ]:
curves = {}
for event_type in event_types:
    if event_type not in available_event_types:
        print(f"skipping {event_type!r} -- not present in this file")
        continue
    enriched_events = load_enriched_events(str(events_path), f"events/{event_type}")
    aggregate = aggregate_events(
        enriched_events, event_type=event_type, days_per_bin=days_per_bin
    )
    curves[event_type] = epidemic_curve(aggregate)
    print(f"{event_type}: {int(curves[event_type]['count'].sum())} events "
          f"over {len(aggregate.bin_starts)} bins, "
          f"{len(aggregate.geo_unit_ids)} geo units")

## Plot
matplotlib is imported **here**, in the consumer -- never in `core`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for event_type, curve in curves.items():
    ax.plot(curve["bin_start"], curve["count"], label=event_type)
ax.set_xlabel(f"time (bins of {days_per_bin} day)")
ax.set_ylabel("events per bin")
ax.set_title("Epidemic curves (events-only)")
ax.legend()
fig.tight_layout()
plt.show()

## Optional: export an aggregate to CSV
`to_long_dataframe` materialises just the slice you want (a tidy `[bin_start, geo_unit_id, event_type, count]` frame) -- a reusable intermediate, CSV-ready.

In [ ]:
from core.aggregate import to_long_dataframe

if "infections" in available_event_types:
    enriched_infections = load_enriched_events(str(events_path), "events/infections")
    infections_aggregate = aggregate_events(
        enriched_infections, event_type="infections", days_per_bin=days_per_bin
    )
    long_frame = to_long_dataframe(infections_aggregate)
    long_frame.head()